# 08.6d Stable Audio Open 推理入口

Stable Audio Open 代表开放但 gated 的工业级扩散音乐模型。接受模型许可、登录 Hugging Face，并准备好依赖后，本 Notebook 会通过官方 `stable-audio-tools` 直接生成音频。

本 Notebook 是设备策略的一个例外：Stable Audio Open 在 Diffusers/CPU/MPS 组合中可能出现 latent 为 NaN、最终音频静音的问题。runner 会使用 `cuda -> cpu`，跳过 MPS；通用训练 Notebook 和其他模型仍按 `cuda -> mps -> cpu` 检测。


## 运行环境与安装

在 Jupyter 中选择 kernel：`Python 3.10 (chapter08-stable-audio)`。Stable Audio Open 的官方 `stable-audio-tools` 当前面向 Python 3.10；不要把它装进 `venv_ch08_diffusers`。mac/Linux 终端使用：

Hugging Face 当前 CLI 命令名为 `hf`。若 `hf --help` 不可用，可按官方文档先安装 standalone CLI；mac/Linux 使用 `curl -LsSf https://hf.co/cli/install.sh | bash`，Windows PowerShell 使用 `powershell -ExecutionPolicy ByPass -c "irm https://hf.co/cli/install.ps1 | iex"`。也可以把下面的 `hf download ...` 改成 `uvx hf download ...`。

```bash
cd CODE
python3.10 -m venv venv_ch08_stable_audio
source venv_ch08_stable_audio/bin/activate
python -m pip install --upgrade pip wheel
python -m pip install "setuptools<81"
python -m pip install "numpy==1.26.4" "torch==2.7.1" "torchaudio==2.7.1" "torchvision==0.22.1"
python -m pip install "stable-audio-tools==0.0.20"
python -m pip install --force-reinstall "numpy==1.26.4" "PyWavelets==1.4.1" "scipy==1.11.4" "pandas==2.2.3" "librosa==0.10.2.post1" soundfile "torch==2.7.1" "torchaudio==2.7.1" "torchvision==0.22.1" "setuptools<81"
python -m pip install "pytorch-lightning==2.5.5" ipykernel ipywidgets
python -c "import numpy, pywt, scipy, pandas, librosa, torch, torchaudio, torchvision, pytorch_lightning; print('Stable Audio deps ok', numpy.__version__, pywt.__version__, torch.__version__, torchaudio.__version__, pytorch_lightning.__version__)"
python -m ipykernel install --user --name chapter08-stable-audio --display-name "Python 3.10 (chapter08-stable-audio)"
hf download stabilityai/stable-audio-open-1.0 --local-dir chapter08/models/stabilityai_stable_audio_open_1_0
hf download t5-base --local-dir chapter08/models/t5_base
```

Windows PowerShell 使用：

```powershell
cd CODE
py -3.10 -m venv venv_ch08_stable_audio
.\venv_ch08_stable_audio\Scripts\Activate.ps1
python -m pip install --upgrade pip wheel
python -m pip install "setuptools<81"
python -m pip install "numpy==1.26.4" "torch==2.7.1" "torchaudio==2.7.1" "torchvision==0.22.1"
python -m pip install "stable-audio-tools==0.0.20"
python -m pip install --force-reinstall "numpy==1.26.4" "PyWavelets==1.4.1" "scipy==1.11.4" "pandas==2.2.3" "librosa==0.10.2.post1" soundfile "torch==2.7.1" "torchaudio==2.7.1" "torchvision==0.22.1" "setuptools<81"
python -m pip install "pytorch-lightning==2.5.5" ipykernel ipywidgets
python -c "import numpy, pywt, scipy, pandas, librosa, torch, torchaudio, torchvision, pytorch_lightning; print('Stable Audio deps ok', numpy.__version__, pywt.__version__, torch.__version__, torchaudio.__version__, pytorch_lightning.__version__)"
python -m ipykernel install --user --name chapter08-stable-audio --display-name "Python 3.10 (chapter08-stable-audio)"
hf download stabilityai/stable-audio-open-1.0 --local-dir chapter08/models/stabilityai_stable_audio_open_1_0
hf download t5-base --local-dir chapter08/models/t5_base
```

`stable-audio-tools` 是模型卡推荐的主运行时。本章早期 Diffusers 路径在 CPU/MPS 上可能 1 步采样就返回 NaN latent，表现为进度跑完但音频静音；因此 08.6d 改用官方工具栈。CUDA 可用时会使用 CUDA；Apple Silicon 或普通 CPU 会走 CPU，速度明显慢，但不会再经过 Diffusers 的 `CosineDPMSolverMultistepScheduler`。

不要安装 `stable-audio-tools[all]`、`stable-audio-tools[train]` 或 `stable-audio-tools[ui]`。本 Notebook 只需要推理核心依赖；这些 extras 可能拉入与本章无关的 CUDA/训练依赖。

若已经建好环境后出现 `ValueError: numpy.dtype size changed`，说明 `PyWavelets` 的二进制扩展和当前 NumPy ABI 不一致。激活 `venv_ch08_stable_audio` 后执行：

```bash
python -m pip install --force-reinstall "numpy==1.26.4" "PyWavelets==1.4.1" "scipy==1.11.4" "pandas==2.2.3" "librosa==0.10.2.post1" soundfile "torch==2.7.1" "torchaudio==2.7.1" "torchvision==0.22.1" "setuptools<81"
python -m pip install "pytorch-lightning==2.5.5" ipykernel ipywidgets
```

然后重启 Jupyter kernel，再运行本 Notebook。

`pytorch-lightning` 这里不是为了训练 Stable Audio Open，而是因为 `stable-audio-tools==0.0.20` 在构造推理模型时会导入 LoRA callback 类；缺少它会在创建模型时提示缺少 `pytorch_lightning`。
如果 `torchaudio` 报 `libtorchaudio.so` 不能加载，通常是 `torch` 与 `torchaudio` 版本不一致；执行上面三条修复命令会把它们重新锁回同一组版本。


## 模型权重下载

先打开 <https://huggingface.co/stabilityai/stable-audio-open-1.0> 接受许可，再用同一个 Hugging Face 账号登录 CLI。mac/Linux 终端：

```bash
cd CODE
source venv_ch08_stable_audio/bin/activate
hf auth login
hf download stabilityai/stable-audio-open-1.0 --local-dir chapter08/models/stabilityai_stable_audio_open_1_0
hf download t5-base --local-dir chapter08/models/t5_base
```

Windows PowerShell：

```powershell
cd CODE
.\venv_ch08_stable_audio\Scripts\Activate.ps1
hf auth login
hf download stabilityai/stable-audio-open-1.0 --local-dir chapter08/models/stabilityai_stable_audio_open_1_0
hf download t5-base --local-dir chapter08/models/t5_base
```

下载后本 Notebook 会自动优先使用 `chapter08/models/stabilityai_stable_audio_open_1_0` 和 `chapter08/models/t5_base`。

Stable Audio Open 本地目录需要包含 `model_config.json` 以及 `model.safetensors` 或 `model.ckpt`。`stable-audio-tools` 的 T5 conditioner 需要 tokenizer 与 T5 encoder 位于同一个 `from_pretrained` 目录，因此本 Notebook 使用 `chapter08/models/t5_base` 作为本地文本编码器目录。runner 使用官方 `stable-audio-tools` API 加载这些文件，不会在采样时再联网下载模型权重。

Stable Audio Open 的输出波形可能幅度很低。runner 会打印生成张量 peak/RMS、请求窗口 peak/RMS 与保存窗口起点，并按模型卡示例做 peak normalize 后再保存 WAV，避免默认 16-bit WAV 写盘时被量化成静音。


In [ ]:
from pathlib import Path
import os
import sys

# 路径推断：从 cwd 向上找含 CODE/chapter08/_common 的目录；ROOT 指向 CODE/chapter08/
_p = Path.cwd()
while not (_p / "CODE" / "chapter08" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter08/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
ROOT = _p / "CODE" / "chapter08"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Audio, display

from _common.config import load_yaml_config
from _common.device_utils import choose_device
from _common.paths import portable_path
from evaluation.comparison_table import append_model_comparison
from model_runners.base import GenerationRequest
from model_runners.conditioning import build_conditioning_rows

OUTPUT_TABLES = ROOT / "outputs" / "tables"
OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)

def rel(path):
    return portable_path(path, ROOT)

def resolve_device(config=None):
    requested = os.getenv("CHAPTER08_DEVICE")
    if requested is None and config is not None:
        requested = str(config.get("device", "auto"))
    return choose_device(requested or "auto")

def print_setup_guidance(status):
    print(status.reason)
    if status.next_action:
        print(status.next_action)
    print("After completing the setup or moving to compatible hardware, rerun this Notebook; it will load and run the model directly.")

def print_runtime_guidance(error):
    print(str(error))
    print("Resolve the message above, then rerun this Notebook or the current cell.")

from model_runners.stable_audio_open import StableAudioOpenRunner

runner = StableAudioOpenRunner()
status = runner.check_environment()
config = load_yaml_config(ROOT / "configs" / "stable_audio_open_inference.yaml")
display(pd.DataFrame([status.as_row()]))


In [ ]:
display(pd.DataFrame(build_conditioning_rows("stable_audio_open")))
condition_row = {
    "prompt": config["prompts"][0]["text"],
    "duration_seconds": config.get("duration_seconds", ""),
    "seed": 8,
    "num_inference_steps": config.get("num_inference_steps", ""),
    "guidance_scale": config.get("guidance_scale", ""),
    "sampler_type": config.get("sampler_type", "dpmpp-3m-sde"),
}
display(pd.DataFrame([condition_row]))


In [ ]:
prompt = config["prompts"][0]
request = GenerationRequest(
    prompt=prompt["text"],
    prompt_id=prompt["prompt_id"],
    duration_sec=float(config.get("duration_seconds", 12)),
    output_dir=ROOT / config["outputs"]["audio_dir"],
    seed=8,
    extra={
        "model_name": os.getenv("CHAPTER08_STABLE_AUDIO_MODEL", config.get("model_name", runner.model_name)),
        "device": resolve_device(config),
        "num_inference_steps": int(config.get("num_inference_steps", 50)),
        "guidance_scale": float(config.get("guidance_scale", 7.0)),
        "sampler_type": config.get("sampler_type", "dpmpp-3m-sde"),
        "sigma_min": float(config.get("sigma_min", 0.3)),
        "sigma_max": float(config.get("sigma_max", 500.0)),
    },
)

if status.available:
    try:
        result = runner.generate(request)
    except RuntimeError as exc:
        print_runtime_guidance(exc)
    else:
        append_model_comparison(
            ROOT / config["outputs"]["table_csv"],
            {
                "model_name": result.model_name,
                "prompt_id": result.prompt_id,
                "dataset_context": "text prompt plus negative prompt",
                "duration_sec": result.duration_sec,
                "wall_time_sec": result.wall_time_sec,
                "device": result.device,
                "output_audio_path": result.output_audio_path,
            },
        )
        display(Audio(str(result.output_audio_path)))
else:
    print_setup_guidance(status)
